# Building a Google Sheets Manager Agent

This tutorial shows how to build a practical agent that manages Google Sheets using the Google Sheets API.

## What This Agent Does:
- Opens and navigates between sheets
- Reads cell values and ranges
- Updates cells with values or formulas
- Inserts and deletes rows/columns
- Sorts data by column
- Merges cells
- Filters and searches for data

## Steps:
1. Set up Google Sheets API credentials
2. Define Sheet actions (direct gspread API wrappers)
3. Create the SheetManagerAgent
4. Add examples to guide the agent
5. Test with real spreadsheet operations

**Note:** This is a practical implementation - no environment wrapper or reward system! Just direct spreadsheet operations. 🎯

## Setup: Install Dependencies and Configure API

### Prerequisites:
1. Create a Google Cloud Project
2. Enable Google Sheets API
3. Create a Service Account and download credentials JSON
4. Save credentials as `credentials.json` in your project directory

See: https://docs.gspread.org/en/latest/oauth2.html#for-bots-using-service-account

In [ ]:
# Install required packages if needed
# !pip install gspread google-auth google-auth-oauthlib google-auth-httplib2 agentlite

import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Path to your service account credentials
CREDENTIALS_FILE = os.getenv("GOOGLE_SHEETS_CREDENTIALS", "credentials.json")

# Check if credentials file exists
if not os.path.exists(CREDENTIALS_FILE):
    print("⚠️  Warning: credentials.json not found")
    print("Please download your service account credentials from Google Cloud Console")
    print("and save it as credentials.json")
else:
    print("✅ Google Sheets credentials found")

## Import Sheet Actions Module

Using modular structure - actions are defined in separate files, similar to benchmarks!

**Structure:**
```
sheet_actions/
  ├── __init__.py          # Package exports
  ├── sheet_actions.py     # Action implementations (OpenSheet, UpdateCell, etc.)
  └── sheet_agent.py       # SheetManagerAgent definition
```

In [ ]:
# Import from the sheet_actions module
from sheet_actions import get_gspread_client
from sheet_actions.sheet_agent import SheetManagerAgent

# Initialize gspread client
try:
    gc = get_gspread_client(CREDENTIALS_FILE)
    print("✅ Successfully connected to Google Sheets API")
    print("\nSheet actions available (used by agent):")
    print("  - OpenSpreadsheet: Open a spreadsheet by name/ID")
    print("  - OpenSheet: Switch to a specific worksheet")
    print("  - GetAllValues: Read entire sheet")
    print("  - GetCellValue: Read single cell")
    print("  - UpdateCell: Update single cell")
    print("  - InsertRows: Add new rows")
    print("  - FindCell: Search for values")
    print("  - SortSheetByColumn: Sort data")
    print("  - GetSheetSummary: Get sheet overview")
except Exception as e:
    print(f"❌ Error connecting to Google Sheets: {e}")

## Test the Agent

Now let's test with real spreadsheet operations!

**Note:** You'll need to:
1. Create a test spreadsheet in Google Sheets
2. Share it with the service account email (found in your credentials.json)
3. Replace 'YOUR_SPREADSHEET_NAME' with your actual spreadsheet name or ID

In [ ]:
from agentlite.commons import TaskPackage

# Test 1: Open a spreadsheet and get summary
print("=" * 60)
print("TEST 1: Open spreadsheet and get summary")
print("=" * 60)

task1 = "Open the spreadsheet named 'Test Sheet' and give me a summary of Sheet1"
task_pack1 = TaskPackage(instruction=task1)
response1 = sheet_agent(task_pack1)
print(f"\n📊 Response: {response1}\n")

In [ ]:
# Test 2: Read specific cell values
print("=" * 60)
print("TEST 2: Read cell values")
print("=" * 60)

task2 = "What is the value in cell A1 of the current sheet?"
task_pack2 = TaskPackage(instruction=task2)
response2 = sheet_agent(task_pack2)
print(f"\n📋 Response: {response2}\n")

In [ ]:
# Test 3: Update a cell
print("=" * 60)
print("TEST 3: Update a cell value")
print("=" * 60)

task3 = "Update cell B2 to contain the text 'Updated by Agent'"
task_pack3 = TaskPackage(instruction=task3)
response3 = sheet_agent(task_pack3)
print(f"\n✏️  Response: {response3}\n")

In [ ]:
# Test 4: Find a value
print("=" * 60)
print("TEST 4: Find a cell containing specific text")
print("=" * 60)

task4 = "Find the cell that contains 'Total' and tell me its location"
task_pack4 = TaskPackage(instruction=task4)
response4 = sheet_agent(task_pack4)
print(f"\n🔍 Response: {response4}\n")

## Example: Complex Workflow - Data Analysis

In [ ]:
# Create a more complex workflow
print("=" * 60)
print("COMPLEX WORKFLOW: Analyze and update data")
print("=" * 60)

task_workflow = """In the current sheet, find all cells in column A that contain 'Product',
then read the values in column B next to them, and give me a summary."""

task_pack_workflow = TaskPackage(instruction=task_workflow)
response_workflow = sheet_agent(task_pack_workflow)
print(f"\n📊 Response: {response_workflow}\n")

## Direct API Testing (Without Agent)

You can also use the actions directly without the agent for simple operations:

In [ ]:
# Direct usage examples
print("Direct Action Testing:\n")

# Create action instances
open_spreadsheet = OpenSpreadsheet()
open_sheet = OpenSheet()
get_summary = GetSheetSummary()

# Use them directly
result1 = open_spreadsheet("Test Sheet")
print(f"1. {result1}\n")

result2 = open_sheet("Sheet1")
print(f"2. {result2}\n")

result3 = get_summary()
print(f"3. {result3}\n")

## Modular Structure vs Benchmark

This tutorial uses a **modular structure** similar to benchmarks but **dramatically simpler**:

### 📁 File Structure:
```
sheet_actions/
  ├── __init__.py          # Package exports
  ├── sheet_actions.py     # 11 action classes + helper functions
  └── sheet_agent.py       # SheetManagerAgent class
```

### ❌ What We DON'T Have (Benchmark-only features):
- **No SheetEnv Class**: No environment wrapper managing state
- **No `step()` Function**: Actions call gspread API directly
- **No Reward System**: No matching score against ground truth
- **No `get_match_score()`**: No comparison with expected results
- **No `update()` method**: No reward calculation
- **No `action_path` tracking**: Simpler execution model

### ✅ What We DO Have (Practical features):
- **Modular Action Files**: Actions organized in `sheet_actions.py`
- **Separate Agent File**: Agent defined in `sheet_agent.py`
- **Clean Imports**: Notebook imports from modules
- **Direct gspread API Calls**: Actions use `gspread` library directly
- **Simple Global State**: `current_spreadsheet` and `current_worksheet` variables
- **Real Spreadsheet Operations**: Actually modifies your Google Sheets
- **Easy to Use**: Straightforward API, no complex abstractions
- **Production Ready**: Can be used in real applications

### Comparison Table:

| Feature | Benchmark (tool-operation) | This Tutorial (Practical) |
|---------|---------------------------|---------------------------|
| File Organization | ✅ Modular structure | ✅ Modular structure |
| Environment Class | ✅ `SheetEnv` | ❌ Not needed |
| `step()` Function | ✅ Returns (obs, reward, done, info) | ❌ Direct execution |
| Reward Calculation | ✅ Compares with ground truth | ❌ Binary success/fail |
| State Management | ✅ Complex tracking | ✅ Simple global vars |
| Action Execution | Via environment wrapper | Direct API calls |
| Use Case | Research/Evaluation | Production Apps |
| Code Complexity | High (~500+ lines) | Low (~200 lines) |

### When to Use Each Pattern:

**Use Benchmark Pattern (with Environment + Rewards) when:**
- Comparing agent performance quantitatively
- Need reproducible evaluation metrics
- Testing against known correct answers
- Academic research

**Use Modular Practical Pattern (this tutorial) when:**
- Building real user applications
- Automating spreadsheet tasks
- Integration with business workflows
- Simplicity and maintainability matter
- **This is what you want 99% of the time!** 🎯

## Summary

This tutorial demonstrates a **production-ready, modular** Google Sheets agent that:
- Uses organized module structure like benchmarks
- Imports cleanly into notebooks
- Uses the gspread library directly (no wrapper complexity)
- Provides intelligent natural language interface to spreadsheets
- Can be easily extended with more actions
- Is maintainable and debuggable

The benchmark version's environment/reward system is **only for research evaluation**, not for practical use! 🚀